# 从房价问题重建

先在纸上写每一步的问题与矩阵，再执行核对。价格模型与所有观测都为教学假设。

依赖：NumPy、Matplotlib。按顺序运行，先在纸上预测再核对；图内使用英文标签。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 由两套房找系数

未知量是两个系数，价格单位万元，面积平方米，距离公里。先消元，不要先运行。


In [ ]:
A2 = np.array([[80.,1.],[100.,2.]])
b2 = np.array([150.,180.])
coefficients = np.linalg.solve(A2,b2)
print('系数:',coefficients)
assert np.allclose(coefficients,[2,-10])


## 反过来预测

数据行 (90,0.5) 与系数相乘，结果代表什么？


In [ ]:
new_house = np.array([90.,.5])
predicted_price = new_house @ coefficients
print('模型预测（万元）:',predicted_price)
assert np.allclose(predicted_price,175)


## 新观测让精确解失败

第三套房的观测是 180。最佳系数、预测与残差分别是什么？


In [ ]:
A = np.vstack((A2,new_house))
b = np.array([150.,180.,180.])
fit,_,rank,_ = np.linalg.lstsq(A,b,rcond=None)
residual = b-A @ fit
print('最佳系数:',fit,'预测:',A @ fit,'残差:',residual,'秩:',rank)
assert np.linalg.norm(residual)>0
assert np.allclose(A.T @ residual,0,atol=1e-8)
Q,R = np.linalg.qr(A,mode='reduced')
fit_qr = np.linalg.solve(R,Q.T @ b)
assert np.allclose(fit_qr,fit)


## 加一列重复单位

第三列是距离米数，第二列是公里数。新系数的预测是否增加独立能力？


In [ ]:
redundant = np.column_stack((A,A[:,1]*1000))
z = np.array([0.,-1000.,1.])
assert np.allclose(redundant @ z,0)
shortest = np.linalg.pinv(redundant) @ b
print('新矩阵秩:',np.linalg.matrix_rank(redundant),'最短系数:',shortest)
assert np.linalg.matrix_rank(redundant)==2
assert np.allclose(redundant @ shortest,A @ fit,atol=1e-8)
for t in [0.,.01,-.01]:
    changed = shortest+t*z
    assert np.allclose(redundant @ changed,A @ fit,atol=1e-8)
    print('另一组系数:',changed,'预测:',redundant @ changed)


## 用 SVD 再读一次结构

哪些奇异值非零？各个方向属于输入还是输出？


In [ ]:
U,s,Vt = np.linalg.svd(redundant,full_matrices=False)
print('奇异值:',s)
assert np.allclose((U*s) @ Vt,redundant)
print('重建误差:',np.linalg.norm((U*s) @ Vt-redundant))
print('请用列空间、零空间、投影解释上面每个结果。')


## 合上实验，自己解释

指出已知量、未知量、单位与计算目的。用纸上推导解释观察结果，再改变一个参数检查；有限次计算不是一般结论的证明。
